# 跨境电商多站点批量预测 - SageMaker Batch Transform

适用于大规模批量预测场景（数千个 SKU）

In [ ]:
!pip install -U -q "sagemaker<3" pandas

In [ ]:
import sys
sys.path.append("code_preprocess")

import json
import boto3
import tempfile
import tarfile
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
from sagemaker import Session
from sagemaker.model import Model
from sagemaker.jumpstart.model import JumpStartModel
from sagemaker.transformer import Transformer
from preprocess import preprocess_data, PAST_COVARIATES, FUTURE_COVARIATES

## 1. 配置

In [ ]:
ROLE = None
PREDICTION_LENGTH = 28
FREQ = "D"
MARKETPLACES = ["US", "UK", "DE"]
ITEMS_PER_BATCH = 100

## 2. 创建 SageMaker Model

In [ ]:
def repackage_model(js_model, bucket, key):
    s3 = boto3.client("s3")
    s3_uri = js_model.model_data["S3DataSource"]["S3Uri"].rstrip("/") + "/"
    src_bucket, prefix = s3_uri.replace("s3://", "").split("/", 1)
    
    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        for page in s3.get_paginator("list_objects_v2").paginate(Bucket=src_bucket, Prefix=prefix):
            for obj in page.get("Contents", []):
                if not obj["Key"].endswith("/"):
                    local = tmpdir / obj["Key"][len(prefix):]
                    local.parent.mkdir(parents=True, exist_ok=True)
                    s3.download_file(src_bucket, obj["Key"], str(local))
        
        tar_path = tmpdir / "model.tar.gz"
        with tarfile.open(tar_path, "w:gz") as tar:
            tar.add(tmpdir, arcname=".")
        s3.upload_file(str(tar_path), bucket, key)
    
    return f"s3://{bucket}/{key}"

In [ ]:
session = Session()
bucket = session.default_bucket()
s3_prefix = "ecommerce-chronos"

js_model = JumpStartModel(
    model_id="pytorch-forecasting-chronos-2",
    instance_type="ml.c5.4xlarge",
    role=ROLE,
)

model_uri = repackage_model(js_model, bucket, f"{s3_prefix}/model.tar.gz")
print(f"模型已上传: {model_uri}")

In [ ]:
chronos_model = Model(
    name="chronos-2-ecommerce",
    model_data=model_uri,
    image_uri=js_model.image_uri,
    role=ROLE,
)
chronos_model.create()

## 3. 准备批量数据

In [ ]:
def prepare_batch_input(df, marketplace, bucket, s3_prefix, 
                        prediction_length=28, items_per_batch=100):
    df = preprocess_data(df, marketplace=marketplace)
    
    inputs = []
    for asin, group in df.sort_values(["asin", "date"]).groupby("asin"):
        entry = {
            "target": group["sales_quantity"].tolist(),
            "item_id": str(asin),
            "start": group["date"].iloc[0].isoformat(),
        }
        
        valid_past = [c for c in PAST_COVARIATES if c in group.columns]
        if valid_past:
            entry["past_covariates"] = {c: group[c].fillna(0).tolist() for c in valid_past}
        
        last_date = group["date"].max()
        future_dates = pd.date_range(last_date + timedelta(days=1), periods=prediction_length)
        future_df = preprocess_data(
            pd.DataFrame({"date": future_dates, "asin": asin, "sales_quantity": 0}),
            marketplace=marketplace
        )
        valid_future = [c for c in FUTURE_COVARIATES if c in future_df.columns]
        if valid_future:
            entry["future_covariates"] = {c: future_df[c].fillna(0).tolist() for c in valid_future}
        
        inputs.append(entry)
    
    params = {"prediction_length": prediction_length, "freq": "D", "quantile_levels": [0.1, 0.5, 0.9]}
    lines = []
    for i in range(0, len(inputs), items_per_batch):
        batch = inputs[i:i + items_per_batch]
        lines.append(json.dumps({"inputs": batch, "parameters": params}))
    
    s3 = boto3.client("s3")
    key = f"{s3_prefix}/batch-input/{marketplace}/input.jsonl"
    s3.put_object(Bucket=bucket, Key=key, Body="\n".join(lines).encode())
    
    return f"s3://{bucket}/{key}", len(inputs)

In [ ]:
# ========== 替换为真实数据 ==========
import numpy as np
np.random.seed(42)

for marketplace in MARKETPLACES:
    dates = pd.date_range("2024-01-01", "2024-12-31")
    asins = [f"B0{marketplace}{i:03d}" for i in range(50)]
    
    data = []
    for asin in asins:
        base = np.random.randint(30, 150)
        for date in dates:
            seasonal = 3.0 if date.month in [10, 11, 12] else 1.0
            sales = int(base * seasonal * np.random.uniform(0.7, 1.3))
            data.append({"asin": asin, "date": date, "sales_quantity": sales})
    
    df = pd.DataFrame(data)
    input_uri, count = prepare_batch_input(df, marketplace, bucket, s3_prefix)
    print(f"{marketplace}: {count} 产品, 数据已上传到 {input_uri}")

## 4. 执行 Batch Transform

In [ ]:
def run_batch_transform(model_name, input_uri, output_uri, instance_type="ml.c5.4xlarge"):
    transformer = Transformer(
        model_name=model_name,
        instance_count=1,
        instance_type=instance_type,
        output_path=output_uri,
        strategy="SingleRecord",
        assemble_with="Line",
        accept="application/json",
    )
    transformer.transform(
        data=input_uri,
        content_type="application/json",
        split_type="Line",
        wait=True,
    )
    return transformer

In [ ]:
for marketplace in MARKETPLACES:
    input_uri = f"s3://{bucket}/{s3_prefix}/batch-input/{marketplace}/input.jsonl"
    output_uri = f"s3://{bucket}/{s3_prefix}/batch-output/{marketplace}/"
    
    print(f"\n开始 {marketplace} 站点批量预测...")
    run_batch_transform(chronos_model.name, input_uri, output_uri)
    print(f"{marketplace} 完成")

## 5. 获取结果

In [ ]:
def load_batch_results(bucket, s3_prefix, marketplace):
    s3 = boto3.client("s3")
    key = f"{s3_prefix}/batch-output/{marketplace}/input.jsonl.out"
    result = s3.get_object(Bucket=bucket, Key=key)
    lines = result["Body"].read().decode().strip().split("\n")
    
    all_preds = []
    for line in lines:
        preds = json.loads(line)["predictions"]
        all_preds.extend(preds)
    
    dfs = []
    for pred in all_preds:
        df = pd.DataFrame({
            "asin": pred.get("item_id"),
            "date": pd.date_range(pred["start"], periods=len(pred["mean"]), freq="D"),
            "forecast": pred["mean"],
            "lower_10": pred["0.1"],
            "upper_90": pred["0.9"],
            "marketplace": marketplace,
        })
        dfs.append(df)
    
    return pd.concat(dfs, ignore_index=True)

In [ ]:
all_forecasts = []
for marketplace in MARKETPLACES:
    forecast_df = load_batch_results(bucket, s3_prefix, marketplace)
    all_forecasts.append(forecast_df)
    print(f"{marketplace}: {forecast_df['asin'].nunique()} 产品")

final_df = pd.concat(all_forecasts, ignore_index=True)
final_df.to_csv("forecast_all_markets.csv", index=False)
print(f"\n总计: {len(final_df)} 条预测记录")
final_df.head()